# TensorFlow YOLO Object Detection with Kubeflow Trainer

This example demonstrates how to train a YOLOv3-tiny object detection model using TensorFlow and scale it with Kubeflow TrainJob for distributed training.

## Overview

- **Model**: YOLOv3-tiny (lightweight YOLO variant)
- **Dataset**: COCO dataset (subset for demonstration)
- **Framework**: TensorFlow 2.x with Keras API
- **Distribution**: TensorFlow MultiWorkerMirroredStrategy

## Install Dependencies

Install the required packages for TensorFlow and Kubeflow SDK:

In [ ]:
# Uncomment to install dependencies
# !pip install -U kubeflow tensorflow==2.15.0 tensorflow-datasets opencv-python-headless numpy

## Define the Training Function

Create a training function that implements YOLOv3-tiny with TensorFlow distributed training support:

In [ ]:
def train_yolo_object_detection(
    num_epochs=10,
    batch_size=16,
    learning_rate=0.001,
    image_size=416,
    num_classes=80,
):
    """Train YOLOv3-tiny model for object detection using TensorFlow.
    
    Args:
        num_epochs: Number of training epochs
        batch_size: Batch size per device
        learning_rate: Learning rate for optimizer
        image_size: Input image size (square)
        num_classes: Number of object classes (COCO has 80)
    """
    import json
    import os
    
    import numpy as np
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    
    print(f"TensorFlow version: {tf.__version__}")
    print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")
    
    # Setup distributed training strategy
    # TF_CONFIG is automatically set by Kubeflow Trainer for distributed training
    tf_config = os.environ.get('TF_CONFIG')
    
    if tf_config:
        print(f"TF_CONFIG: {tf_config}")
        strategy = tf.distribute.MultiWorkerMirroredStrategy()
        print(f"Number of devices: {strategy.num_replicas_in_sync}")
    else:
        # Single device training
        if tf.config.list_physical_devices('GPU'):
            strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
        else:
            strategy = tf.distribute.OneDeviceStrategy(device="/cpu:0")
        print("Running on single device")
    
    # Adjust batch size for distributed training
    global_batch_size = batch_size * strategy.num_replicas_in_sync
    print(f"Global batch size: {global_batch_size}")
    
    # Define YOLOv3-tiny architecture
    def build_yolov3_tiny(input_shape, num_classes):
        """Build YOLOv3-tiny model architecture."""
        inputs = keras.Input(shape=input_shape)
        
        # Backbone: Feature extraction layers
        x = layers.Conv2D(16, 3, padding='same', activation='leaky_relu')(inputs)
        x = layers.MaxPooling2D(2, strides=2, padding='same')(x)
        
        x = layers.Conv2D(32, 3, padding='same', activation='leaky_relu')(x)
        x = layers.MaxPooling2D(2, strides=2, padding='same')(x)
        
        x = layers.Conv2D(64, 3, padding='same', activation='leaky_relu')(x)
        x = layers.MaxPooling2D(2, strides=2, padding='same')(x)
        
        x = layers.Conv2D(128, 3, padding='same', activation='leaky_relu')(x)
        x = layers.MaxPooling2D(2, strides=2, padding='same')(x)
        
        x = layers.Conv2D(256, 3, padding='same', activation='leaky_relu')(x)
        route_1 = x  # Save for skip connection
        x = layers.MaxPooling2D(2, strides=2, padding='same')(x)
        
        x = layers.Conv2D(512, 3, padding='same', activation='leaky_relu')(x)
        x = layers.MaxPooling2D(2, strides=1, padding='same')(x)
        
        x = layers.Conv2D(1024, 3, padding='same', activation='leaky_relu')(x)
        
        # Detection head 1 (for larger objects)
        x = layers.Conv2D(256, 1, padding='same', activation='leaky_relu')(x)
        route_2 = x
        x = layers.Conv2D(512, 3, padding='same', activation='leaky_relu')(x)
        
        # Output layer 1: (batch, grid, grid, anchors * (5 + num_classes))
        # 5 = x, y, w, h, confidence
        output_1 = layers.Conv2D(3 * (5 + num_classes), 1, padding='same')(x)
        
        # Detection head 2 (for smaller objects)
        x = layers.Conv2D(128, 1, padding='same', activation='leaky_relu')(route_2)
        x = layers.UpSampling2D(2)(x)
        x = layers.Concatenate()([x, route_1])
        x = layers.Conv2D(256, 3, padding='same', activation='leaky_relu')(x)
        
        # Output layer 2
        output_2 = layers.Conv2D(3 * (5 + num_classes), 1, padding='same')(x)
        
        model = keras.Model(inputs=inputs, outputs=[output_1, output_2])
        return model
    
    # Create synthetic dataset for demonstration
    # In production, replace with actual COCO dataset loading
    def create_synthetic_dataset(batch_size, image_size, num_samples=1000):
        """Create synthetic dataset for demonstration."""
        def generator():
            for _ in range(num_samples):
                # Random images
                image = np.random.rand(image_size, image_size, 3).astype(np.float32)
                
                # Random labels (simplified)
                # In real YOLO: [grid_h, grid_w, anchors, (x, y, w, h, conf, classes)]
                label_1 = np.random.rand(13, 13, 3, 5 + num_classes).astype(np.float32)
                label_2 = np.random.rand(26, 26, 3, 5 + num_classes).astype(np.float32)
                
                yield image, (label_1, label_2)
        
        dataset = tf.data.Dataset.from_generator(
            generator,
            output_signature=(
                tf.TensorSpec(shape=(image_size, image_size, 3), dtype=tf.float32),
                (
                    tf.TensorSpec(shape=(13, 13, 3, 5 + num_classes), dtype=tf.float32),
                    tf.TensorSpec(shape=(26, 26, 3, 5 + num_classes), dtype=tf.float32),
                )
            )
        )
        
        dataset = dataset.batch(batch_size)
        dataset = dataset.prefetch(tf.data.AUTOTUNE)
        return dataset
    
    # Custom YOLO loss function (simplified)
    def yolo_loss(y_true, y_pred):
        """Simplified YOLO loss function."""
        # In production, implement proper YOLO loss with:
        # - Localization loss (MSE for bbox coordinates)
        # - Confidence loss (binary cross-entropy)
        # - Classification loss (categorical cross-entropy)
        return tf.reduce_mean(tf.square(y_true - y_pred))
    
    # Build and compile model within strategy scope
    with strategy.scope():
        model = build_yolov3_tiny(
            input_shape=(image_size, image_size, 3),
            num_classes=num_classes
        )
        
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        
        model.compile(
            optimizer=optimizer,
            loss=[yolo_loss, yolo_loss],
            metrics=['accuracy']
        )
    
    print("\nModel Summary:")
    model.summary()
    
    # Create dataset
    train_dataset = create_synthetic_dataset(
        batch_size=batch_size,
        image_size=image_size,
        num_samples=1000
    )
    
    # Setup callbacks
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath='./checkpoints/yolo_epoch_{epoch:02d}.h5',
            save_freq='epoch',
            verbose=1
        ),
        keras.callbacks.TensorBoard(
            log_dir='./logs',
            histogram_freq=1
        ),
    ]
    
    # Train the model
    print("\nStarting training...")
    history = model.fit(
        train_dataset,
        epochs=num_epochs,
        callbacks=callbacks,
        verbose=1
    )
    
    # Save final model
    if not tf_config or json.loads(tf_config)['task']['index'] == 0:
        model.save('./yolo_final_model.h5')
        print("\nModel saved to ./yolo_final_model.h5")
    
    print("\nTraining completed!")
    print(f"Final loss: {history.history['loss'][-1]:.4f}")
    
    return history

## Run Training Locally

Test the training function locally before scaling to distributed training:

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient, LocalProcessBackendConfig

# Initialize local backend
backend_config = LocalProcessBackendConfig(cleanup_venv=True)
client = TrainerClient(backend_config=backend_config)

# List available runtimes
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":  # Use torch-distributed runtime
        selected_runtime = runtime
        break

In [ ]:
# Submit local training job
job_name = client.train(
    trainer=CustomTrainer(
        func=train_yolo_object_detection,
        func_args={
            "num_epochs": 2,
            "batch_size": 8,
            "learning_rate": 0.001,
            "image_size": 416,
        },
        packages_to_install=["tensorflow==2.15.0", "numpy"],
    ),
    runtime=selected_runtime,
)

# Stream logs
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end='')

## Scale with Kubeflow TrainJob

Now scale the training to multiple nodes using Kubeflow Trainer:

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient

# Initialize Kubeflow client
client = TrainerClient()

# List available runtimes
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

### Submit Distributed Training Job

Train the YOLO model across multiple nodes:

In [ ]:
job_name = client.train(
    trainer=CustomTrainer(
        func=train_yolo_object_detection,
        func_args={
            "num_epochs": 10,
            "batch_size": 16,
            "learning_rate": 0.001,
            "image_size": 416,
            "num_classes": 80,
        },
        # Set number of training nodes
        num_nodes=3,
        # Set resources per node
        resources_per_node={
            "cpu": 4,
            "memory": "16Gi",
            # Uncomment for GPU training
            # "nvidia.com/gpu": 1,
        },
    ),
    runtime=torch_runtime,
)

print(f"TrainJob created: {job_name}")

### Check TrainJob Status

Monitor the training job components:

In [ ]:
# Wait for running status
client.wait_for_job_status(name=job_name, status={"Running"})

# Check individual steps
for step in client.get_job(name=job_name).steps:
    print(f"Step: {step.name}, Status: {step.status}, Devices: {step.device} x {step.device_count}")

### Watch Training Logs

Stream logs from the training job:

In [ ]:
for logline in client.get_job_logs(job_name, follow=True):
    print(logline)

### Verify Completion

Wait for the training job to complete:

In [ ]:
client.wait_for_job_status(name=job_name, timeout=600)
print(f"TrainJob {job_name} completed successfully!")

### Cleanup

Delete the TrainJob when finished:

In [ ]:
# Uncomment to delete the job
# client.delete_job(job_name)

## Next Steps

To use this example with real data:

1. **Load COCO Dataset**: Replace synthetic data with actual COCO dataset using `tensorflow_datasets`
2. **Implement Full YOLO Loss**: Add proper localization, confidence, and classification losses
3. **Add Data Augmentation**: Include random flips, crops, color jittering
4. **Enable Mixed Precision**: Use `tf.keras.mixed_precision` for faster training
5. **Add Evaluation**: Implement mAP (mean Average Precision) metrics
6. **Model Checkpointing**: Save best models based on validation metrics

## Resources

- [TensorFlow Distributed Training Guide](https://www.tensorflow.org/guide/distributed_training)
- [YOLO Paper](https://arxiv.org/abs/1506.02640)
- [COCO Dataset](https://cocodataset.org/)
- [Kubeflow Trainer Documentation](https://www.kubeflow.org/docs/components/trainer/)